In [6]:
from google.colab import files
from pathlib import Path
import zipfile, csv
import numpy as np
from scipy.ndimage import distance_transform_edt
from scipy.spatial import cKDTree

# Requires file from notebook 1, ZIP file MMA3001 apple tree dataset
dataset_zip = Path('/content/MMA3001_appletree_dataset.zip')

files.upload()

with zipfile.ZipFile(dataset_zip) as z:
    assert 'inventory.csv' in z.namelist(), 'inventory.csv is missing.'

auth.authenticate_user()
credentials, _ = default()
service = build('drive', 'v3', credentials=credentials)


In [3]:
# Read dataset inventory
with zipfile.ZipFile(dataset_zip) as z:
    inventory = list(csv.DictReader(
        io.StringIO(z.read('inventory.csv').decode('utf-8-sig'))
    ))

# Group depth images by folder, season and tree
groups = {}

for entry in inventory:
    name = entry['name']

    if '_D-' not in name or not name.endswith('.png'):
        continue

    tree, season = name.split('_')[:2]

    if season in ('Summer', 'Winter'):
        key = (entry['source_folder'], season)
        groups.setdefault(key, {}).setdefault(tree, entry)

# Select three trees from each group
selection = []

for (folder, season), trees in sorted(groups.items()):
    for tree in sorted(trees)[:3]:
        selection.append((folder, season, trees[tree]))

print('Selected images:', len(selection))

for folder, season, entry in selection:
    print(folder, season, entry['name'])


Selected images: 12
Dataset Summer R01U01_Summer_D-12-40-11.png
Dataset Summer R01U02_Summer_D-12-40-26.png
Dataset Summer R01U03_Summer_D-12-40-37.png
Dataset Winter R01U01_Winter_D-11-23-20.png
Dataset Winter R01U02_Winter_D-11-23-42.png
Dataset Winter R01U03_Winter_D-11-23-45.png
Dataset (With Summer GT) Summer R01N01_Summer_D-12-40-50.png
Dataset (With Summer GT) Summer R01N02_Summer_D-12-40-59.png
Dataset (With Summer GT) Summer R01N03_Summer_D-12-41-4.png
Dataset (With Summer GT) Winter R01N01_Winter_D-11-23-49.png
Dataset (With Summer GT) Winter R01N02_Winter_D-11-23-52.png
Dataset (With Summer GT) Winter R01N03_Winter_D-11-23-55.png


In [4]:

import time
from scipy.ndimage import distance_transform_edt
from scipy.spatial import cKDTree

# Split depth pixels into training, validation and testing
def make_split(depth, seed=42, block_size=None):
    valid = depth > 0

    if block_size is None:
        positions = np.argwhere(valid)
    else:
        h, w = depth.shape
        positions = [
            (y, x)
            for y in range(0, h - block_size + 1, block_size)
            for x in range(0, w - block_size + 1, block_size)
            if valid[y:y+block_size, x:x+block_size].mean() >= 0.5
        ]
        positions = np.array(positions)

    order = np.random.default_rng(seed).permutation(len(positions))
    n = max(1, int(len(positions) * 0.05))

    val = np.zeros(depth.shape, dtype=bool)
    test = np.zeros(depth.shape, dtype=bool)

    for mask, chosen in [(val, order[:n]), (test, order[n:2*n])]:
        for i in chosen:
            y, x = positions[i]

            if block_size is None:
                mask[y, x] = True
            else:
                mask[y:y+block_size, x:x+block_size] = True

    val &= valid
    test &= valid

    return valid & ~val & ~test, val, test


def make_block_split(depth, block_size=12, seed=42):
    return make_split(depth, seed, block_size)


# Interpolate missing depth values
def interpolate_depth(depth, train, query, method='nearest', radius=6):
    prediction = np.full(depth.shape, np.nan)
    supported = np.zeros(depth.shape, dtype=bool)

    if not train.any() or not query.any():
        return prediction, supported

    if method == 'nearest':
        distances, indices = distance_transform_edt(
            ~train, return_indices=True
        )

        supported = query & (distances <= radius)
        prediction[supported] = depth[
            tuple(index[supported] for index in indices)
        ]

    elif method == 'idw':
        known = np.argwhere(train)
        targets = np.argwhere(query)

        k = min(8, len(known))
        distances, indices = cKDTree(known).query(
            targets, k=k, distance_upper_bound=radius
        )

        if k == 1:
            distances = distances[:, None]
            indices = indices[:, None]

        ok = np.isfinite(distances) & (indices < len(known))

        weights = np.zeros_like(distances)
        weights[ok] = 1 / np.maximum(distances[ok], 1e-10)**2

        nearby = np.zeros_like(weights)
        nearby[ok] = depth[train][indices[ok]]

        total = weights.sum(axis=1)
        found = total > 0
        y, x = targets[found].T

        prediction[y, x] = (
            (nearby[found] * weights[found]).sum(axis=1)
            / total[found]
        )
        supported[y, x] = True

    else:
        raise ValueError('Method must be nearest or idw.')

    return prediction, supported


print('Numerical methods ready.')


Numerical methods ready.


In [5]:

# Compare nearest-neighbour and IDW on the same pixels
result_rows = []

for folder, season, entry in selection:

    # Download depth image
    data = service.files().get_media(
        fileId=entry['id'], supportsAllDrives=True
    ).execute()

    with Image.open(io.BytesIO(data)) as img:
        arr = np.array(img)

    # Convert RGB depth image to grayscale
    if arr.ndim == 3:
        if not (np.array_equal(arr[:, :, 0], arr[:, :, 1]) and
                np.array_equal(arr[:, :, 1], arr[:, :, 2])):
            raise ValueError('Invalid depth image: ' + entry['name'])
        depth = arr[:, :, 0]
    else:
        depth = arr

    # Test both validation designs
    for design in ('random_pixels', 'spatial_blocks_12'):

        if design == 'random_pixels':
            train, val, test = make_split(depth, seed=42)
        else:
            train, val, test = make_block_split(depth, block_size=12, seed=42)

        predictions = {}
        support = {}

        # Run both interpolation methods
        for method in ('nearest', 'idw'):
            pred, ok = interpolate_depth(
                depth, train, test, method=method, radius=6
            )
            predictions[method] = pred
            support[method] = ok & np.isfinite(pred)

        # Pixels predicted by both methods
        shared = test & support['nearest'] & support['idw']

        # Calculate errors on the same pixels
        for method in ('nearest', 'idw'):
            error = predictions[method][shared] - depth[shared]

            result_rows.append({
                'folder': folder,
                'season': season,
                'tree': entry['name'].split('_')[0],
                'image': entry['name'],
                'design': design,
                'method': method,
                'shared_rmse': float(np.sqrt(np.mean(error**2))) if error.size else np.nan,
                'shared_mae': float(np.mean(np.abs(error))) if error.size else np.nan,
                'common_pixels': int(shared.sum()),
                'requested_pixels': int(test.sum()),
                'common_fraction': float(shared.sum() / test.sum()) if test.any() else 0,
                'individual_coverage': float((test & support[method]).sum() / test.sum()) if test.any() else 0
            })

    print('Completed:', entry['name'])

print('Computed', len(result_rows), 'common-pixel metric records')


Completed: R01U01_Summer_D-12-40-11.png
Completed: R01U02_Summer_D-12-40-26.png
Completed: R01U03_Summer_D-12-40-37.png
Completed: R01U01_Winter_D-11-23-20.png
Completed: R01U02_Winter_D-11-23-42.png
Completed: R01U03_Winter_D-11-23-45.png
Completed: R01N01_Summer_D-12-40-50.png
Completed: R01N02_Summer_D-12-40-59.png
Completed: R01N03_Summer_D-12-41-4.png
Completed: R01N01_Winter_D-11-23-49.png
Completed: R01N02_Winter_D-11-23-52.png
Completed: R01N03_Winter_D-11-23-55.png
Computed 48 common-pixel metric records


In [ ]:

import pandas as pd

# Convert results to table
frame = pd.DataFrame(result_rows)

# Save results
filename = 'MMA3001_appletree_commonsupportmetric.csv'
frame.to_csv(filename, index=False)

# Display median results
print(frame.groupby(['design', 'method'])[
    ['shared_rmse', 'shared_mae', 'common_fraction', 'individual_coverage']
].median())

# Download CSV, Produces Excel data for common support metric
files.download(filename)


                           shared_rmse  shared_mae  common_fraction  \
design            method                                              
random_pixels     idw         1.291751    0.380617         1.000000   
                  nearest     1.635163    0.341740         1.000000   
spatial_blocks_12 idw         3.633308    1.145926         0.939323   
                  nearest     4.022191    1.057136         0.939323   

                           individual_coverage  
design            method                        
random_pixels     idw                 1.000000  
                  nearest             1.000000  
spatial_blocks_12 idw                 0.939323  
                  nearest             0.993195  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>